# Lab 06 · Strings y fechas

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crapdev/sparkLab/blob/main/labs/06-strings-fechas/lab.ipynb)

**Objetivo:** Limpiar y combinar texto con upper, initcap, substring y concat_ws, y trabajar con fechas de verdad: to_date, year, datediff, months_between, date_add y date_format.

**Tiempo estimado:** 40 minutos

## Antes de empezar

1. Ve a **Archivo → Guardar una copia en Drive**. Así trabajas sobre tu copia y no pierdes nada.
2. Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
3. Donde veas `...` te toca escribir código. Las celdas de test te dicen si está bien:
   ✅ es correcto y ❌ viene con una pista.
4. Si te atascas, vuelve a la teoría del módulo antes de mirar la solución.

## Preparación

Esta celda instala la versión de PySpark con la que se probó el laboratorio y comprueba
que hay una versión de Java compatible. En Colab tarda alrededor de un minuto la primera vez.

In [ ]:
import importlib.metadata
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.2.0"
MIN_JAVA_VERSION = 17


def installed_pyspark_version():
    try:
        return importlib.metadata.version("pyspark")
    except importlib.metadata.PackageNotFoundError:
        return None


def java_major_version():
    java = shutil.which("java")
    if java is None:
        return 0
    output = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    match = re.search(r'version "(\d+)', output)
    return int(match.group(1)) if match else 0


if installed_pyspark_version() != PYSPARK_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )

if java_major_version() < MIN_JAVA_VERSION:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Spark 4 necesita Java 17 o superior. Instálalo y repite esta celda.")
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"PySpark {PYSPARK_VERSION} y Java {java_major_version()} listos.")

La siguiente celda define `check`, la función que usan los tests. No hace falta que la
modifiques: ejecútala y sigue.

In [ ]:
RESULTS = {}


def check(name, condition, hint):
    """Run one test: `condition` is a function that must return True."""
    try:
        passed = bool(condition())
        error = None
    except Exception as exc:  # A broken answer is a failed test, not a crash.
        passed, error = False, exc
    RESULTS[name] = passed
    if passed:
        print(f"✅ {name}")
        return
    detail = f" ({type(error).__name__}: {error})" if error else ""
    print(f"❌ {name}{detail}")
    print(f"   Pista: {hint}")
    assert passed, f"{name} · {hint}"

## Sesión de Spark

Esta celda crea la `SparkSession` del laboratorio e importa las funciones como `F`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("lab-06-strings-fechas").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo.")

## Datos

Esta celda descarga `bigmart_sales.csv`, `drivers.json` la primera vez y los deja en la carpeta `data/`.
Usa `dataset_path("nombre")` para obtener la ruta de un archivo.

In [ ]:
import os
import urllib.request

DATA_BASE_URLS = [
    "https://raw.githubusercontent.com/crapdev/sparkLab/main/public/data",
]
DATA_DIR = os.environ.get("PYSPARK_LABS_DATA_DIR", "data")
DATASETS = ["bigmart_sales.csv", "drivers.json"]


def dataset_path(name):
    """Return the local path of a dataset, downloading it the first time."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    os.makedirs(DATA_DIR, exist_ok=True)
    for base_url in DATA_BASE_URLS:
        try:
            urllib.request.urlretrieve(f"{base_url}/{name}", path)
            return path
        except OSError:
            continue  # Try the next mirror.
    raise RuntimeError(f"No se pudo descargar {name}. Revisa tu conexión y repite la celda.")


for dataset in DATASETS:
    print("Listo:", dataset_path(dataset))

In [ ]:
sales = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(dataset_path("bigmart_sales.csv"))
)
drivers = spark.read.format("json").load(dataset_path("drivers.json"))

print("Productos:", sales.count(), "· Pilotos:", drivers.count())

## Tarea 1 · Mayúsculas y trozos de texto

Las funciones de texto viven en `F` y devuelven una columna nueva: `F.upper`, `F.lower`,
`F.initcap` (primera letra de cada palabra en mayúscula) y `F.substring(columna, inicio,
longitud)`, donde el inicio cuenta **desde 1**.

Los identificadores de BigMart empiezan por dos letras que indican la familia del
producto: `FD` (comida), `DR` (bebida) y `NC` (no comestible).

Crea `coded` a partir de `sales` con dos cambios:

- una columna nueva `prefix` con las dos primeras letras de `Item_Identifier`
- `Item_Type` sobrescrita en mayúsculas

In [ ]:
coded = ...

In [ ]:
check(
    "Tarea 1 · prefix tiene las dos primeras letras",
    lambda: coded.filter(F.col("prefix") == "FD").count() == 6125 and coded.filter(F.col("prefix") == "DR").count() == 799,
    'Usa F.substring("Item_Identifier", 1, 2): el inicio es 1, no 0. Hay 6125 FD, 1599 NC y 799 DR.',
)
check(
    "Tarea 1 · Item_Type está en mayúsculas",
    lambda: coded.filter(F.col("Item_Type") == "SOFT DRINKS").count() == 445 and "Item_Type" in coded.columns,
    'Sobrescribe la columna: withColumn("Item_Type", F.upper("Item_Type")).',
)
check(
    "Tarea 1 · 13 columnas y 8523 filas",
    lambda: len(coded.columns) == 13 and coded.count() == 8523,
    "prefix se añade e Item_Type se sobrescribe: 12 + 1 columnas, sin perder filas.",
)

Las tres funciones de mayúsculas, lado a lado. Fíjate en `initcap` con `Item_Fat_Content`:
convierte `LF` en `Lf` y `reg` en `Reg`, así que no sirve para unificar esas variantes;
para eso, `lower` y después un `when`, como en el módulo anterior.

In [ ]:
sales.select(F.initcap("Item_Type").alias("initcap"), F.lower("Item_Type").alias("lower"), F.upper("Item_Type").alias("upper")).distinct().show(3, truncate=False)
sales.select(F.initcap("Item_Fat_Content").alias("fat")).groupBy("fat").count().orderBy("fat").show()

## Tarea 2 · Unir textos

`F.concat_ws(separador, col1, col2, ...)` une varias columnas con un separador. Si una es
nula, la salta. `F.concat` hace lo mismo sin separador, pero **si alguna columna es nula,
el resultado es nulo**.

Los pilotos tienen el nombre partido en `name.forename` y `name.surname`. Crea
`named` añadiendo a `drivers` la columna `full_name`: nombre y apellido separados por un
espacio.

In [ ]:
named = ...

In [ ]:
check(
    "Tarea 2 · Lewis Hamilton existe",
    lambda: named.filter(F.col("full_name") == "Lewis Hamilton").count() == 1,
    'Usa F.concat_ws(" ", "name.forename", "name.surname"). El separador va primero.',
)
check(
    "Tarea 2 · ningún nombre nulo",
    lambda: named.filter(F.col("full_name").isNull()).count() == 0 and named.count() == 885,
    "Con concat_ws no hay nulos. Si usaste concat con una columna nula, el resultado sería nulo.",
)

La diferencia con los nulos importa: 765 pilotos no tienen `code`. Mira qué pasa al unir
`driverRef` con `code` de las dos formas:

In [ ]:
print("concat:   ", drivers.select(F.concat("driverRef", F.lit("-"), "code").alias("c")).filter(F.col("c").isNull()).count(), "nulos")
print("concat_ws:", drivers.select(F.concat_ws("-", "driverRef", "code").alias("c")).filter(F.col("c").isNull()).count(), "nulos")
drivers.filter(F.col("code").isNull()).select("driverRef", "code", F.concat_ws("-", "driverRef", "code").alias("concat_ws")).show(2)

## Tarea 3 · De texto a fecha

`dob` (fecha de nacimiento) llegó del JSON como `string`. Mientras sea texto, no puedes
restar fechas ni sacar el año de forma fiable. `F.to_date(columna)` la convierte al tipo
`date`. Si el texto no está en formato `yyyy-MM-dd`, hay que decirle el patrón:
`F.to_date(columna, "dd/MM/yyyy")`.

Crea `dated` a partir de `named` **sobrescribiendo** `dob` con su versión de tipo fecha.

In [ ]:
dated = ...

In [ ]:
check(
    "Tarea 3 · dob es de tipo date",
    lambda: dict(dated.dtypes)["dob"] == "date",
    'Usa withColumn("dob", F.to_date("dob")). El texto ya está en yyyy-MM-dd, no hace falta patrón.',
)
check(
    "Tarea 3 · Hamilton nació el 7 de enero de 1985",
    lambda: str(dated.filter(F.col("driverRef") == "hamilton").first()["dob"]) == "1985-01-07",
    "Convierte la columna dob, no otra.",
)
check(
    "Tarea 3 · los 7 nulos siguen siendo nulos",
    lambda: dated.filter(F.col("dob").isNull()).count() == 7,
    "to_date deja los nulos como están. No rellenes ni filtres.",
)

Un patrón equivocado no devuelve nulos: en Spark 4 **falla al ejecutar**, como el `cast`.
Esta celda intenta leer `1985-01-07` con el patrón `dd/MM/yyyy`:

In [ ]:
try:
    dated.select(F.to_date(F.lit("1985-01-07"), "dd/MM/yyyy")).show()
except Exception as error:
    print(type(error).__name__)
    print(str(error).split("\n")[0])

## Tarea 4 · Partes de una fecha

Con una columna `date`, `F.year`, `F.month` y `F.dayofmonth` sacan cada parte como un
número entero.

Crea `with_year` añadiendo a `dated` la columna `birth_year` con el año de nacimiento, y
`young` con los pilotos nacidos en 1990 o después.

In [ ]:
with_year = ...
young = ...

In [ ]:
check(
    "Tarea 4 · birth_year es un entero y vale 1985 para Hamilton",
    lambda: dict(with_year.dtypes)["birth_year"] == "int"
    and with_year.filter(F.col("driverRef") == "hamilton").first()["birth_year"] == 1985,
    'Usa F.year("dob"). Devuelve un int, no un texto.',
)
check(
    "Tarea 4 · 59 pilotos nacidos en 1990 o después",
    lambda: young.count() == 59,
    "Filtra birth_year >= 1990, incluyendo 1990. Los 7 pilotos sin fecha quedan fuera solos.",
)

## Tarea 5 · Restar fechas

Dos formas de medir la distancia entre fechas:

- `F.datediff(fin, inicio)` devuelve **días**. El orden importa: si `fin` es anterior a
  `inicio`, sale negativo.
- `F.months_between(fin, inicio)` devuelve **meses** con decimales.

Para una edad, lo cómodo es `months_between` dividido entre 12 y redondeado hacia abajo
con `F.floor`. Para que el resultado no cambie cada día, aquí se calcula a una fecha
fija: `REFERENCE`, el 1 de enero de 2025.

Crea `aged` añadiendo a `with_year` la columna `age`: la edad en años cumplidos a
`REFERENCE`, como entero (`.cast("int")` después de `F.floor`).

In [ ]:
REFERENCE = F.to_date(F.lit("2025-01-01"))

aged = ...

In [ ]:
check(
    "Tarea 5 · Hamilton tenía 39 años el 1 de enero de 2025",
    lambda: aged.filter(F.col("driverRef") == "hamilton").first()["age"] == 39,
    "Es F.months_between(REFERENCE, F.col(\"dob\")) / 12, en ese orden, y F.floor para quedarte con los años cumplidos.",
)
check(
    "Tarea 5 · el piloto más joven tenía 16",
    lambda: aged.filter(F.col("age").isNotNull()).agg(F.min("age")).first()[0] == 16,
    "Si te sale negativo, los argumentos de months_between están al revés.",
)
check(
    "Tarea 5 · 39 pilotos con menos de 30 años y age es int",
    lambda: aged.filter(F.col("age") < 30).count() == 39 and dict(aged.dtypes)["age"] == "int",
    'Termina con .cast("int"): floor devuelve un bigint.',
)

Compara con `datediff` y mira el signo según el orden de los argumentos:

In [ ]:
hamilton = aged.filter(F.col("driverRef") == "hamilton")
hamilton.select(
    "dob",
    F.datediff(REFERENCE, F.col("dob")).alias("days_lived"),
    F.datediff(F.col("dob"), REFERENCE).alias("reversed"),
    F.months_between(REFERENCE, F.col("dob")).alias("months"),
    "age",
).show()

Sumar y restar días es más directo: `F.date_add(fecha, n)` y `F.date_sub(fecha, n)`.
Un número negativo en `date_add` también resta. Y `F.current_date()` da la fecha de hoy,
que es lo que usarías en un proceso real en vez de `REFERENCE`.

In [ ]:
hamilton.select(
    "dob",
    F.date_add("dob", 7).alias("week_after"),
    F.date_sub("dob", 7).alias("week_before"),
    F.date_add("dob", -7).alias("also_before"),
    F.current_date().alias("today"),
).show()

## Tarea 6 · De fecha a texto con formato

`F.date_format(fecha, patrón)` convierte una fecha en **texto** con el formato que pidas.
Las letras del patrón importan, mayúsculas incluidas: `yyyy` año, `MM` mes, `dd` día.
`mm` son minutos y `YYYY` ni siquiera se acepta.

Crea `formatted` añadiendo a `aged` la columna `dob_text` con la fecha como `dd/MM/yyyy`.

In [ ]:
formatted = ...

In [ ]:
check(
    "Tarea 6 · Hamilton sale como 07/01/1985",
    lambda: formatted.filter(F.col("driverRef") == "hamilton").first()["dob_text"] == "07/01/1985",
    'El patrón es "dd/MM/yyyy": día, mes en mayúsculas, año en minúsculas.',
)
check(
    "Tarea 6 · dob_text es string y dob sigue siendo date",
    lambda: dict(formatted.dtypes)["dob_text"] == "string" and dict(formatted.dtypes)["dob"] == "date",
    "date_format devuelve texto. Ponlo en una columna nueva y conserva dob como fecha.",
)

Los dos tropiezos clásicos del patrón. `mm` son minutos, así que el mes sale `00`. Y
`YYYY` (año por semanas) ya no se acepta: Spark lanza un error que explica el cambio.

In [ ]:
hamilton.select(F.date_format("dob", "dd-MM-yyyy").alias("MM_mes"), F.date_format("dob", "dd-mm-yyyy").alias("mm_minutos")).show()

try:
    hamilton.select(F.date_format("dob", "dd/MM/YYYY")).show()
except Exception as error:
    print(type(error).__name__)
    print(str(error).split("\n")[0][:200])

## Para terminar · texto que parece fecha

Como `dob` venía en `yyyy-MM-dd`, muchas cosas funcionan incluso sin `to_date`: Spark
convierte el texto por debajo cuando una función de fechas lo necesita. Eso engaña. En
cuanto la fecha llega en otro formato (`07/01/1985`, `Jan 7 1985`), la conversión
implícita falla, y comparar textos como si fueran fechas da resultados absurdos con
formatos distintos. Convierte siempre con `to_date` al leer y trabaja con fechas de verdad.

In [ ]:
print("year() sobre el texto:", drivers.filter(F.col("driverRef") == "hamilton").select(F.year("dob")).first()[0])
print("Esquema con to_date:   ", dict(dated.dtypes)["dob"])

## Resumen

Ejecuta esta celda al final. Si todos los tests pasan, vuelve al sitio y marca el
laboratorio como completado.

In [ ]:
EXPECTED_TESTS = 15
passed_tests = sum(RESULTS.values())

print(f"{passed_tests} de {EXPECTED_TESTS} tests superados")
for test_name, passed in RESULTS.items():
    print("  ✅" if passed else "  ❌", test_name)

assert passed_tests == EXPECTED_TESTS, "Todavía quedan tareas por resolver."
print("\n🎉 ¡Laboratorio completado! Vuelve al sitio y márcalo como hecho.")